# OneClickAI 표 회귀
이 노트북은 Google Colab 환경에서 실행하도록 작성되었습니다.

**순서**
1. 파일 업로드
2. 데이터 전처리 및 모델 학습
3. 결과 확인

## 1. 파일 업로드

엑셀 파일(`.xlsx`, `.xls`)을 업로드해 주세요.  
파일에는 **원인** 시트와 **결과** 시트가 필요합니다.

In [ ]:
# ==========================================
# 파일 업로드
# ==========================================
from google.colab import files
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, Input
import matplotlib.pyplot as plt

print("데이터 파일(.xlsx, .xls)을 업로드 해주세요.\n")
uploaded = files.upload()

if len(uploaded.keys()) == 0:
    raise ValueError("파일이 업로드되지 않았습니다.")

DATA_FILE_PATH = list(uploaded.keys())[0]

if not DATA_FILE_PATH.endswith(('.xls', '.xlsx')):
    raise ValueError("지원하지 않는 파일 형식입니다. .xlsx 또는 .xls 파일을 업로드 해주세요.")


# ==========================================
# 파일 열기
# ==========================================
df_x = pd.read_excel(DATA_FILE_PATH, sheet_name='원인', index_col=0)
df_y = pd.read_excel(DATA_FILE_PATH, sheet_name='결과', index_col=0)
print(f"데이터 로딩 완료: {DATA_FILE_PATH} (원인: {df_x.shape}, 결과: {df_y.shape})\n")


## 2. 데이터 전처리 및 모델 학습

데이터를 정규화하고 학습/검증(8:2)으로 분리한 뒤 회귀 모델을 학습합니다.

In [ ]:

# ==========================================
# 데이터 확인
# ==========================================
varname_x = list(df_x.columns)
varname_y = list(df_y.columns)
len_x, len_y, len_data = len(df_x.columns), len(df_y.columns), len(df_x)
print(f'독립변수: {varname_x}\n')
print(f'종속변수: {varname_y}\n')
print(f'독립변수 {len_x}개, 종속변수 {len_y}개, 데이터 {len_data}개\n')

# ==========================================
# 데이터 전처리 (Min-Max 정규화)
# ==========================================
x = np.array(df_x)
y = np.array(df_y).astype(float)
x = (x - x.min()) / (x.max() - x.min())
y_min, y_max = y.min(), y.max()
y = (y - y_min) / (y_max - y_min)

# ==========================================
# 데이터를 training set과 validation set으로 나누기
# ==========================================
split_idx = int(len_data * 0.8)
x_train, x_val = x[:split_idx], x[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]
print(f'training: {len(x_train)}개, validation: {len(x_val)}개\n')

# ==========================================
# 모델 생성
# ==========================================
model = tf.keras.Sequential()
model.add(Input(shape=(x_train.shape[1],)))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(64, activation='relu'))
model.add(Dense(len_y))
model.summary()

# ==========================================
# 모델 설정 및 학습
# ==========================================
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
history = model.fit(x_train, y_train, validation_data=(x_val, y_val), batch_size=32, epochs=100)

# ==========================================
# 모델 저장 및 불러오기
# ==========================================
model.save('tabular_regression_model.keras')
model_loaded = tf.keras.models.load_model('tabular_regression_model.keras')



## 3. 결과 확인

학습 과정의 손실(MSE) / MAE 그래프를 출력하고, 실제값과 예측값을 비교합니다.

In [ ]:

# ==========================================
# 모델 결과 확인 (그래프)
# ==========================================
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper right')

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'])
plt.plot(history.history['val_mae'])
plt.title('model MAE')
plt.ylabel('MAE')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper right')

plt.show()

# ==========================================
# 테스트 및 시각화
# ==========================================
pred = model.predict(x_val) * (y_max - y_min) + y_min
actual = y_val * (y_max - y_min) + y_min

plt.figure(figsize=(6, 6))
plt.scatter(actual, pred, alpha=0.5)
plt.plot([actual.min(), actual.max()], [actual.min(), actual.max()], 'r--', lw=2)
plt.title('Actual vs Predicted')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.grid(True)
plt.show()

test_idx = 0
print("테스트 데이터 1번 실제값:")
for name, val in zip(varname_y, actual[test_idx]):
    print(f"  {name}: {val}\n")
print("테스트 데이터 1번 예측값:")
for name, val in zip(varname_y, pred[test_idx]):
    print(f"  {name}: {val:.4f}\n")
